In [ ]:
import json
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
from getpass import getpass
API_KEY = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

Enter your Gemini API key: ··········


## Task 4.1

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [ ]:
SYSTEM_TEXT = """
You are an appointment operations analysis assistant.
Analyse clinic appointment notes carefully.
Use only information supported by the note.
Do not invent patient information.
Return valid JSON matching the supplied schema.
"""

# Providing realistic root causes so the model can make a valid classification
ROOT_CAUSE_VALUES = [
    "Transportation",
    "Scheduling Conflict",
    "Financial / Insurance"
]

In [ ]:
from google.genai.errors import APIError
try:
    from google.genai.errors import ServerError
except ImportError:
    ServerError = APIError
import time

def analyse_note(note_text):
    prompt = f"""
    CONTEXT:
    Analyse a clinic appointment note to identify
    possible reasons for a patient missing an appointment.

    ROLE:
    Act as an experienced healthcare appointment
    operations analyst.

    ACTION:
    Identify the most likely root cause and recommend
    a practical action.

    Allowed root_cause values:
    {ROOT_CAUSE_VALUES}

    Use exactly one of these values.
    Do not create a new category.

    TONE:
    Be objective, professional and concise.
    Do not make unsupported assumptions.

    FORMAT:
    Return the four required JSON fields:
    root_cause, confidence, summary, recommended_action.

    Confidence must be between 0 and 1.
    Summary must contain no more than 25 words.

    APPOINTMENT NOTE:
    {note_text}
    """

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT,
                    temperature=0,
                    max_output_tokens=1200,
                    response_mime_type="application/json",
                    response_schema=SCHEMA
                )
            )

            if not response.text:
                raise ValueError("Gemini returned an empty response.")

            result = json.loads(response.text)

            required_fields = {
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            }

            if set(result.keys()) != required_fields:
                raise ValueError("Unexpected JSON fields.")

            if not isinstance(result["root_cause"], str):
                raise ValueError("root_cause must be a string.")

            if not 0 <= result["confidence"] <= 1:
                raise ValueError("Confidence must be between 0 and 1.")

            if len(result["summary"].split()) > 25:
                raise ValueError("Summary exceeds 25 words.")

            if result["root_cause"] not in ROOT_CAUSE_VALUES:
                raise ValueError("Invalid root_cause category.")

            return result

        except ServerError as e:
            if "503" in str(e) and attempt < 2:
                wait_time = 2 ** (attempt + 1)
                print(f"Gemini server busy. Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                raise

In [ ]:
result_N001 = analyse_note(note_N001)
print(json.dumps(result_N001, indent=2))

## Task 4.2

In [ ]:
class NoteAnalysis(BaseModel):

    root_cause: str
    confidence: float = Field(ge=0, le=1)
    summary: str
    recommended_action: str

    @field_validator("summary")
    @classmethod
    def check_summary_words(cls, value):
        if len(value.split()) > 25:
            raise ValueError("Summary must contain at most 25 words")
        return value

In [ ]:
valid_reply = {
    "root_cause": "transport_issue",
    "confidence": 0.85,
    "summary": "Patient reported difficulty arranging transport.",
    "recommended_action": "Call the patient and offer assistance."
}

In [ ]:
bad_reply_1 = valid_reply.copy()
bad_reply_1["confidence"] = 1.7

try:
    NoteAnalysis.model_validate(bad_reply_1)

except ValidationError as e:
    print("Validation Error 1:")
    print(e)

In [ ]:
bad_reply_2 = valid_reply.copy()
bad_reply_2.pop("recommended_action")

try:
    NoteAnalysis.model_validate(bad_reply_2)

except ValidationError as e:
    print("\nValidation Error 2:")
    print(e)

In [ ]:
ROOT_CAUSE_VALUES = [
    # Replace these examples with your actual course categories
    "transport_issue",
    "forgot_appointment",
    "scheduling_conflict",
    "financial_issue",
    "health_issue",
    "other"
]

In [ ]:
note_N007 = notes_df.loc[
    notes_df["note_id"] == "N007",
    "note_text"
].iloc[0]

note_N009 = notes_df.loc[
    notes_df["note_id"] == "N009",
    "note_text"
].iloc[0]

print("N007:", note_N007)
print("N009:", note_N009)

In [ ]:
def build_v1_prompt(note_text):

    return f"""
    Classify this note:

    {note_text}
    """

In [ ]:
def build_v2_prompt(note_text):

    return f"""
    CONTEXT:
    You are analysing clinic appointment notes to identify
    possible reasons for a patient missing an appointment.

    ROLE:
    Act as an experienced healthcare appointment
    operations analyst.

    ACTION:
    Classify the root cause of the appointment note.

    Allowed root_cause values:
    {ROOT_CAUSE_VALUES}

    Select exactly one value from the allowed list.
    If the note is ambiguous, unclear, or does not provide
    enough evidence, answer "other" rather than guessing.

    TONE:
    Be objective, careful and professional.
    Do not invent information.

    FORMAT:
    Return JSON containing:
    root_cause, confidence, summary, recommended_action.

    Confidence must be between 0 and 1.
    Summary must contain no more than 25 words.

    APPOINTMENT NOTE:
    {note_text}
    """

In [ ]:
API_CALLS_4_3 = 0

def get_gemini_reply(prompt):

    global API_CALLS_4_3

    for attempt in range(3):

        try:
            API_CALLS_4_3 += 1

            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT,
                    temperature=0,
                    max_output_tokens=1200,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                    thinking_config=types.ThinkingConfig(
                        thinking_level="low"
                    )
                )
            )

            if not response.text:
                raise ValueError("Empty Gemini response")

            return json.loads(response.text)

        except ServerError as e:

            if "503" in str(e) and attempt < 2:
                wait_time = 2 ** (attempt + 1)
                print(f"Server busy. Retrying in {wait_time} seconds.")
                time.sleep(wait_time)

            else:
                raise

## Task 4.4

In [ ]:
notes = {
    "N007": note_N007,
    "N009": note_N009
}

results = []

for note_id, note_text in notes.items():

    # V1
    v1_prompt = build_v1_prompt(note_text)
    v1_result = get_gemini_reply(v1_prompt)

    results.append({
        "Note": note_id,
        "Prompt": "V1",
        "Root Cause": v1_result["root_cause"],
        "Confidence": v1_result["confidence"],
        "Allowed": v1_result["root_cause"] in ROOT_CAUSE_VALUES
    })

    # V2
    v2_prompt = build_v2_prompt(note_text)
    v2_result = get_gemini_reply(v2_prompt)

    results.append({
        "Note": note_id,
        "Prompt": "V2",
        "Root Cause": v2_result["root_cause"],
        "Confidence": v2_result["confidence"],
        "Allowed": v2_result["root_cause"] in ROOT_CAUSE_VALUES
    })

In [ ]:
comparison_df = pd.DataFrame(results)

print(comparison_df.to_string(index=False))

In [ ]:
calls_task_41 = 1
calls_task_42 = 9
calls_task_43 = API_CALLS_4_3

total_api_calls = (
    calls_task_41 +
    calls_task_42 +
    calls_task_43
)

print("API calls in Task 4.3:", calls_task_43)
print("Total API calls across Tasks 4.1–4.3:", total_api_calls)

In [ ]:
notes = pd.read_csv("notes_analysed.csv")
appointments = pd.read_csv("appointments_clean.csv")

print("Notes shape:", notes.shape)
print("Appointments shape:", appointments.shape)

print("\nNotes columns:")
print(notes.columns.tolist())

print("\nAppointment columns:")
print(appointments.columns.tolist())

In [ ]:
merged = pd.merge(
    notes,
    appointments,
    on="appointment_id",
    how="left",
    validate="one_to_one"
)

print("Merged shape:", merged.shape)

merged.head()

In [ ]:
print("Unmatched appointments:", merged["reminder_sent"].isna().sum())

In [ ]:
cross_tab = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"],
    margins=True
)

print(cross_tab)

In [ ]:
cross_tab.to_csv("root_cause_reminder_crosstab.csv")

In [ ]:
cross_tab_percentage = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"],
    normalize="index"
) * 100

print(cross_tab_percentage.round(2))

In [ ]:
root_cause_counts = merged["root_cause"].value_counts()

print("Root cause counts:")
print(root_cause_counts)

most_common_cause = root_cause_counts.index[0]
most_common_count = root_cause_counts.iloc[0]

print("\nMost common root cause:", most_common_cause)
print("Number of notes:", most_common_count)

In [ ]:
github_url = "PASTE_YOUR_PUBLIC_GITHUB_REPOSITORY_LINK_HERE"

part4_finding = (
    f"The most common root cause was {most_common_cause}, "
    f"appearing in {most_common_count} of the "
    f"{len(merged)} analysed notes."
)

readme_content = f"""# NoShowGuard

**GitHub Repository:** {github_url}

## Project Overview

NoShowGuard is a machine learning project designed to analyse
appointment No-Shows and help clinics identify appointments
that may require follow-up.

## Project Findings

1. **Part 1 — Data Cleaning:** Started with 1,020 appointment
   rows, removed 20 duplicate rows and retained 990 cleaned records.

2. **Part 2 — SQL Analysis:** The overall No-Show rate was
   22.02% across the 990 cleaned appointments.

3. **Part 3 — Machine Learning:** Logistic Regression achieved
   a test ROC-AUC of 0.789, using a classification threshold of 0.35.

4. **Part 4 — Note Analysis:** {part4_finding}

## Business Recommendation

The dataset contained 990 cleaned appointments, with an overall
No-Show rate of 22.02%. The Logistic Regression Pipeline was
selected because it achieved a test ROC-AUC of 0.789. A threshold
of 0.35 was selected using training out-of-fold predictions,
resulting in 49 flagged appointments out of 198 test appointments.
On the test set, precision and recall were both 57.14%.
The clinic should use the risk scores to prioritise follow-up calls,
while monitoring the number of calls the front desk can make in
an evening and the cost of leaving appointment slots empty.

## Selected Model

**Logistic Regression Pipeline**

Reasons for selection:
- Highest 5-fold cross-validated ROC-AUC among the four candidates:
  0.7737.
- Test ROC-AUC: 0.7889.
- Selected threshold: 0.35.
- Test precision: 57.14%.
- Test recall: 57.14%.

The threshold was selected using out-of-fold training predictions,
not by tuning against the test set.

## Cross-Tabulation

The root-cause categories were compared against reminder status
using the 10 analysed appointment notes.

See `root_cause_reminder_crosstab.csv` for the results.

## Limitations

1. The 10 notes are an illustration and cannot show the real
   reasons for No-Shows across the entire clinic population.

2. The cleaning statistics came from all rows before the train/test
   split, so the evaluation may contain some preprocessing leakage.

3. The data are observational, so associations between appointment
   features and No-Shows do not establish causation.

## Files

See the repository files for the cleaned dataset, SQL analysis,
machine learning notebook, analysed appointment notes and
risk-score output.

## Disclaimer

This is an educational capstone project. Model predictions should
support, not replace, clinic staff decisions.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content)

print("README.md created successfully!")

In [ ]:
import os

for filename in os.listdir("."):
    if os.path.isfile(filename):
        print(filename)

In [ ]:
gitignore_content = """
.env
*.env
secrets.json
*api_key*
__pycache__/
.ipynb_checkpoints/
"""

with open(".gitignore", "w") as file:
    file.write(gitignore_content)

In [ ]:
import re
from pathlib import Path

patterns = [
    r"AIza[0-9A-Za-z_-]{20,}",
    r"(?i)api[_-]?key\s*=\s*['\"][^'\"]+['\"]"
]

for path in Path(".").glob("*.py"):
    content = path.read_text(errors="ignore")

    for pattern in patterns:
        if re.search(pattern, content):
            print("Possible secret found in:", path.name)

In [ ]:
github_url = "PASTE_YOUR_PUBLIC_GITHUB_REPOSITORY_LINK_HERE"